# US Revenue Forecast — 조회 · 분석 · 시각화 노트북

**소스 테이블** : `us_revenue_forecast_data`

| 셀 | 단계 |
|---|---|
| Cell 1 | 환경 설정 & 경로 자동 감지 |
| Cell 2 | 라이브러리 Import & DB 연결 |
| Cell 3 | 파라미터 설정 |
| Cell 4 | 단일 티커 예측치 조회 함수 |
| Cell 5 | 단일 티커 조회 테스트 + 증감률 표 |
| Cell 6 | Best-N 기업 추출 함수 |
| Cell 7 | Best-N 추출 테스트 |
| Cell 8 | 단일 티커 매출 전망 차트 (Line + Bar) |
| Cell 9 | Best-N 성장률 비교 막대 차트 (4Q / 8Q) |


## Cell 1 · 환경 설정 & 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",       # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",  # 데스크탑
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve()
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root (자동 감지): {root}")
            return root
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root (후보 경로): {candidate}")
            return candidate
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS를 수정하세요.")

_root = _setup_path()
print(f"[확인] sys.path[0] = {sys.path[0]}")


[PATH] root (자동 감지): C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] sys.path[0] = C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\미국전기업_매출_예측


## Cell 2 · 라이브러리 Import & DB 연결

In [2]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from sqlalchemy import text

# 한글 폰트 설정 (Windows)
matplotlib.rcParams["font.family"]      = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

from DATA.config import get_db_info, get_engine, log

db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("[OK] DB 연결 성공")
except Exception as e:
    print(f"[FAIL] DB 연결 실패: {e}")


[OK] DB 연결 성공


## Cell 3 · 파라미터 설정

여기서 테이블명·항목·모델·색상 등을 자유롭게 변경하세요.


In [3]:
# ── 테이블 / 항목 ────────────────────────────────────────
DEST_TABLE = "us_revenue_forecast_data"
ITEM       = "sale"

# ── 사용 가능한 모델 목록 ─────────────────────────────────
# actual / SARIMA / ETS / Prophet / LSTM / Theta / Ensemble
AVAILABLE_MODELS = ["SARIMA", "ETS", "Prophet", "LSTM", "Theta", "Ensemble"]

# ── 차트 스타일 ───────────────────────────────────────────
ACTUAL_COLOR   = "#2C3E50"   # 실제값 색
FORECAST_COLOR = "#E74C3C"   # 예측값 색
BAR_PALETTE    = [           # Best-N 차트 팔레트
    "#2ECC71","#3498DB","#9B59B6","#F39C12","#E74C3C",
    "#1ABC9C","#E67E22","#34495E","#E91E63","#00BCD4",
    "#8BC34A","#FF5722","#607D8B","#795548","#9C27B0",
    "#03A9F4","#CDDC39","#FF9800","#673AB7","#009688",
]

print("[OK] 파라미터 설정 완료")
print(f"     ITEM={ITEM}  |  TABLE={DEST_TABLE}")


[OK] 파라미터 설정 완료
     ITEM=sale  |  TABLE=us_revenue_forecast_data


## Cell 4 · 단일 티커 예측치 조회 함수

`fetch_forecast(ticker, model)` : DB에서 actual + forecast 데이터를 가져와  
기간별 QoQ 증감률과 전체 누적 증감률을 계산해 반환합니다.


In [4]:
def fetch_forecast(
    ticker: str,
    model: str,
    item: str = ITEM,
    forecast_date: str = None,          # None → 가장 최근 forecast_date 자동 선택
) -> dict:
    """
    us_revenue_forecast_data 에서 ticker + model 기준으로
    actual / forecast 시계열을 조회하고 증감률을 계산합니다.

    Returns
    -------
    dict
        actual_df   : actual 시계열 DataFrame  (date, value)
        forecast_df : forecast 시계열 DataFrame (date, value)
        growth_df   : 예측 구간 QoQ 증감률 + 누적 증감률 DataFrame
        forecast_date : 실제 사용된 forecast_date
        ticker / model / item
    """
    # forecast_date 자동 선택
    if forecast_date is None:
        with engine.connect() as conn:
            row = conn.execute(text(f"""
                SELECT MAX(forecast_date)
                FROM   {DEST_TABLE}
                WHERE  ticker = :t AND item = :i
            """), {"t": ticker, "i": item}).fetchone()
        if row is None or row[0] is None:
            raise ValueError(f"[{ticker}] '{item}' 데이터가 없습니다.")
        forecast_date = str(row[0])

    # actual 조회
    with engine.connect() as conn:
        actual_df = pd.read_sql(text(f"""
            SELECT date, value
            FROM   {DEST_TABLE}
            WHERE  ticker        = :t
              AND  item          = :i
              AND  model         = 'actual'
              AND  forecast_date = :fd
            ORDER  BY date
        """), conn, params={"t": ticker, "i": item, "fd": forecast_date})

    # forecast 조회
    with engine.connect() as conn:
        forecast_df = pd.read_sql(text(f"""
            SELECT date, value
            FROM   {DEST_TABLE}
            WHERE  ticker        = :t
              AND  item          = :i
              AND  model         = :m
              AND  data_type     = 'forecast'
              AND  forecast_date = :fd
            ORDER  BY date
        """), conn, params={"t": ticker, "i": item, "m": model, "fd": forecast_date})

    if forecast_df.empty:
        raise ValueError(f"[{ticker}] model='{model}' 예측 데이터가 없습니다.")

    actual_df["date"]   = pd.to_datetime(actual_df["date"])
    forecast_df["date"] = pd.to_datetime(forecast_df["date"])

    # ── 증감률 계산 ──────────────────────────────────────
    # 기준값: actual 마지막 분기 값
    base_value = float(actual_df["value"].iloc[-1])

    growth_rows = []
    prev_val    = base_value

    for i, row in forecast_df.iterrows():
        val        = float(row["value"])
        qoq        = (val - prev_val) / abs(prev_val) * 100 if prev_val != 0 else None
        cum_growth = (val - base_value) / abs(base_value) * 100 if base_value != 0 else None
        growth_rows.append({
            "date"            : row["date"],
            "value"           : round(val, 0),
            "QoQ_pct"         : round(qoq, 2)        if qoq        is not None else None,
            "cumulative_pct"  : round(cum_growth, 2) if cum_growth is not None else None,
        })
        prev_val = val

    growth_df = pd.DataFrame(growth_rows)

    # 전체 기간 누적 증감률 (마지막 예측값 기준)
    total_growth = (float(forecast_df["value"].iloc[-1]) - base_value) / abs(base_value) * 100

    return {
        "ticker"        : ticker,
        "model"         : model,
        "item"          : item,
        "forecast_date" : forecast_date,
        "base_value"    : base_value,
        "total_growth"  : round(total_growth, 2),
        "actual_df"     : actual_df,
        "forecast_df"   : forecast_df,
        "growth_df"     : growth_df,
    }

print("[OK] fetch_forecast 함수 정의 완료")


[OK] fetch_forecast 함수 정의 완료


## Cell 5 · 단일 티커 조회 테스트 + 증감률 표

`TEST_TICKER` 와 `TEST_MODEL` 을 원하는 값으로 변경하세요.


In [5]:
TEST_TICKER = "AAPL"
TEST_MODEL  = "Ensemble"   # SARIMA / ETS / Prophet / LSTM / Theta / Ensemble

result = fetch_forecast(TEST_TICKER, TEST_MODEL)

print(f"ticker        : {result['ticker']}")
print(f"model         : {result['model']}")
print(f"forecast_date : {result['forecast_date']}")
print(f"기준값(마지막 actual): {result['base_value']:,.0f}")
print(f"전체 기간 누적 증감률  : {result['total_growth']:+.2f}%")
print()

# ── 예측치 + 증감률 표 ────────────────────────────────────
print("=" * 62)
print(f"{'날짜':<14} {'예측값':>22} {'QoQ(%)':>10} {'누적증감률(%)':>12}")
print("-" * 62)
for _, row in result["growth_df"].iterrows():
    qoq_str = f"{row['QoQ_pct']:+.2f}%" if row["QoQ_pct"] is not None else "  -"
    cum_str = f"{row['cumulative_pct']:+.2f}%" if row["cumulative_pct"] is not None else "  -"
    print(f"{str(row['date'].date()):<14} {row['value']:>22,.0f} {qoq_str:>10} {cum_str:>12}")
print("=" * 62)


ticker        : AAPL
model         : Ensemble
forecast_date : 2026-03-24
기준값(마지막 actual): 143,756,000,000
전체 기간 누적 증감률  : +13.80%

날짜                                예측값     QoQ(%)     누적증감률(%)
--------------------------------------------------------------
2026-06-30            120,658,440,713    -16.07%      -16.07%
2026-09-30            129,466,426,110     +7.30%       -9.94%
2026-12-31            182,393,300,080    +40.88%      +26.88%
2027-03-31            148,088,231,212    -18.81%       +3.01%
2027-06-30            133,123,792,416    -10.11%       -7.40%
2027-09-30            142,885,076,854     +7.33%       -0.61%
2027-12-31            199,797,603,130    +39.83%      +38.98%
2028-03-31            163,588,809,071    -18.12%      +13.80%


## Cell 6 · Best-N 기업 추출 함수

`get_best_n(model, n, horizon)` :  
지정 모델의 향후 `horizon` 분기 누적 매출 성장률 기준으로  
성장률이 높은 상위 N개 기업을 반환합니다.


In [ ]:
def get_best_n(
    model: str,
    n: int = 20,
    horizon: int = 8,
    item: str = ITEM,
    forecast_date: str = None,
) -> pd.DataFrame:
    """
    us_revenue_forecast_data 에서 model 기준 향후 horizon 분기
    누적 성장률 상위 N 기업을 반환합니다.

    Parameters
    ----------
    model        : 예측 모델명 (SARIMA/ETS/Prophet/LSTM/Theta/Ensemble)
    n            : 추출할 기업 수 (default 20)
    horizon      : 평가 분기 수 (default 8)
    item         : 재무 항목
    forecast_date: None → 최신 forecast_date 자동 선택

    Returns
    -------
    pd.DataFrame
        columns: ticker, base_value, last_forecast,
                 growth_4q_pct, growth_8q_pct (horizon>=8),
                 growth_total_pct (horizon 기준)
    """
    # 최신 forecast_date 자동 선택
    if forecast_date is None:
        with engine.connect() as conn:
            row = conn.execute(text(f"""
                SELECT MAX(forecast_date)
                FROM   {DEST_TABLE}
                WHERE  item = :i AND model = :m AND data_type = 'forecast'
            """), {"i": item, "m": model}).fetchone()
        if row is None or row[0] is None:
            raise ValueError(f"model='{model}' 예측 데이터가 없습니다.")
        forecast_date = str(row[0])

    print(f"[INFO] forecast_date = {forecast_date}  model = {model}  horizon = {horizon}Q")

    # 예측 데이터 전체 조회 (horizon 개 분기만)
    with engine.connect() as conn:
        fc_df = pd.read_sql(text(f"""
            SELECT ticker, date, value
            FROM   {DEST_TABLE}
            WHERE  item          = :i
              AND  model         = :m
              AND  data_type     = 'forecast'
              AND  forecast_date = :fd
            ORDER  BY ticker, date
        """), conn, params={"i": item, "m": model, "fd": forecast_date})

    # actual 마지막 값 조회 (기준값)
    with engine.connect() as conn:
        act_df = pd.read_sql(text(f"""
            SELECT ticker, MAX(date) AS last_date, value
            FROM   {DEST_TABLE}
            WHERE  item          = :i
              AND  model         = 'actual'
              AND  forecast_date = :fd
            GROUP  BY ticker, value
            HAVING date = MAX(date)
        """), conn, params={"i": item, "fd": forecast_date})
        # 위 쿼리가 MariaDB 버전에 따라 동작 안 할 수 있어 Python 단에서 처리
        act_df2 = pd.read_sql(text(f"""
            SELECT ticker, date, value
            FROM   {DEST_TABLE}
            WHERE  item          = :i
              AND  model         = 'actual'
              AND  forecast_date = :fd
            ORDER  BY ticker, date
        """), conn, params={"i": item, "fd": forecast_date})

    # actual 마지막 값 추출
    act_df2["date"] = pd.to_datetime(act_df2["date"])
    base_map = (
        act_df2.sort_values("date")
               .groupby("ticker")["value"]
               .last()
               .to_dict()
    )

    fc_df["date"]  = pd.to_datetime(fc_df["date"])
    fc_df["value"] = fc_df["value"].astype(float)

    rows = []
    for ticker, grp in fc_df.groupby("ticker"):
        grp = grp.sort_values("date").reset_index(drop=True)
        base = base_map.get(ticker)
        if base is None or base == 0:
            continue

        # horizon 제한
        grp_h = grp.head(horizon)
        if len(grp_h) < 1:
            continue

        last_val     = float(grp_h["value"].iloc[-1])
        growth_total = (last_val - base) / abs(base) * 100

        # 4분기 성장률
        grp_4 = grp.head(4)
        growth_4q = None
        if len(grp_4) >= 4:
            v4 = float(grp_4["value"].iloc[-1])
            growth_4q = round((v4 - base) / abs(base) * 100, 2)

        # 8분기 성장률
        grp_8 = grp.head(8)
        growth_8q = None
        if len(grp_8) >= 8:
            v8 = float(grp_8["value"].iloc[-1])
            growth_8q = round((v8 - base) / abs(base) * 100, 2)

        rows.append({
            "ticker"          : ticker,
            "base_value"      : round(base, 0),
            "last_forecast"   : round(last_val, 0),
            "growth_4q_pct"   : growth_4q,
            "growth_8q_pct"   : growth_8q,
            "growth_total_pct": round(growth_total, 2),
        })

    df_out = (
        pd.DataFrame(rows)
          .sort_values("growth_total_pct", ascending=False)
          .head(n)
          .reset_index(drop=True)
    )
    df_out.index += 1   # 1부터 시작하는 순위
    return df_out, forecast_date

print("[OK] get_best_n 함수 정의 완료")


## Cell 7 · Best-N 추출 테스트

In [ ]:
BEST_MODEL   = "Ensemble"   # 평가 기준 모델
BEST_N       = 20           # 추출 기업 수
BEST_HORIZON = 8            # 평가 분기 수

best_df, used_fd = get_best_n(
    model   = BEST_MODEL,
    n       = BEST_N,
    horizon = BEST_HORIZON,
)

print(f"\n[매출 성장률 Top-{BEST_N}]  model={BEST_MODEL}  horizon={BEST_HORIZON}Q")
print(f"forecast_date = {used_fd}")
print()
display(best_df.style
    .format({
        "base_value"      : "{:,.0f}",
        "last_forecast"   : "{:,.0f}",
        "growth_4q_pct"   : "{:+.2f}%",
        "growth_8q_pct"   : "{:+.2f}%",
        "growth_total_pct": "{:+.2f}%",
    })
    .background_gradient(subset=["growth_total_pct"], cmap="YlGn")
    .set_caption(f"매출 성장률 Top-{BEST_N} ({BEST_MODEL}, {BEST_HORIZON}Q)")
)


## Cell 8 · 단일 티커 매출 전망 차트

Line 차트(전체 흐름) + Bar 차트(예측 구간)를 나란히 표시합니다.  
각 데이터 포인트마다 값이 표시됩니다.


In [ ]:
def plot_ticker_forecast(
    ticker: str,
    model: str,
    item: str = ITEM,
    forecast_date: str = None,
    figsize: tuple = (18, 7),
):
    """
    단일 티커의 actual + forecast 를
    (1) 전체 Line 차트  (2) 예측 구간 Bar 차트로 시각화합니다.
    """
    res = fetch_forecast(ticker, model, item, forecast_date)
    act = res["actual_df"].copy()
    fc  = res["forecast_df"].copy()

    act["date"] = pd.to_datetime(act["date"])
    fc["date"]  = pd.to_datetime(fc["date"])

    act_labels = [d.strftime("%YQ%q") if hasattr(d, 'strftime') else str(d)
                  for d in act["date"]]
    fc_labels  = [d.strftime("%YQ%q") if hasattr(d, 'strftime') else str(d)
                  for d in fc["date"]]

    # 분기 레이블 생성 (YYYYQN 형식)
    def qtr_label(dt):
        q = (dt.month - 1) // 3 + 1
        return f"{dt.year}Q{q}"

    act_labels = [qtr_label(d) for d in act["date"]]
    fc_labels  = [qtr_label(d) for d in fc["date"]]

    # 값 단위 자동 결정 (억 / 십억)
    max_val = max(act["value"].max(), fc["value"].max())
    if max_val >= 1e12:
        div, unit = 1e12, "T"
    elif max_val >= 1e9:
        div, unit = 1e9, "B"
    elif max_val >= 1e6:
        div, unit = 1e6, "M"
    else:
        div, unit = 1, ""

    act_vals = act["value"] / div
    fc_vals  = fc["value"]  / div

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(
        f"{ticker}  |  {item.upper()} Forecast  |  model: {model}  |  forecast_date: {res['forecast_date']}",
        fontsize=14, fontweight="bold", y=1.01
    )

    # ── (1) Line 차트 ─────────────────────────────────────
    ax1 = axes[0]
    all_labels = act_labels + fc_labels
    all_vals   = list(act_vals) + list(fc_vals)
    split_idx  = len(act_vals)

    ax1.plot(range(split_idx), act_vals, color=ACTUAL_COLOR,
             linewidth=2.2, marker="o", markersize=5, label="Actual")
    ax1.plot(range(split_idx - 1, len(all_vals)), all_vals[split_idx - 1:],
             color=FORECAST_COLOR, linewidth=2.2, linestyle="--",
             marker="o", markersize=5, label="Forecast")
    ax1.axvline(x=split_idx - 1, color="gray", linestyle=":", linewidth=1.2, alpha=0.7)

    # 값 표시
    for i, v in enumerate(act_vals):
        ax1.annotate(f"{v:.1f}{unit}", (i, v),
                     textcoords="offset points", xytext=(0, 7),
                     ha="center", fontsize=7, color=ACTUAL_COLOR)
    for i, v in enumerate(fc_vals):
        xi = split_idx + i
        ax1.annotate(f"{v:.1f}{unit}", (xi, v),
                     textcoords="offset points", xytext=(0, 7),
                     ha="center", fontsize=7, color=FORECAST_COLOR, fontweight="bold")

    ax1.set_xticks(range(len(all_labels)))
    ax1.set_xticklabels(all_labels, fontsize=7)
    ax1.set_title("전체 흐름 (Actual + Forecast)", fontsize=11)
    ax1.set_ylabel(f"Revenue ({unit})", fontsize=10)
    ax1.legend(fontsize=9)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    ax1.grid(axis="y", linestyle="--", alpha=0.4)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    # ── (2) Bar 차트 (예측 구간) ─────────────────────────
    ax2 = axes[1]
    x   = range(len(fc_vals))
    bars = ax2.bar(x, fc_vals, color=FORECAST_COLOR, alpha=0.85, width=0.6, edgecolor="white")

    # 값 표시
    for bar, v in zip(bars, fc_vals):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(fc_vals) * 0.012,
            f"{v:.1f}{unit}", ha="center", va="bottom",
            fontsize=8, fontweight="bold", color=FORECAST_COLOR
        )

    # QoQ 증감률 표시 (막대 하단)
    gdf = res["growth_df"]
    for i, row in gdf.iterrows():
        if row["QoQ_pct"] is not None:
            color = "#27AE60" if row["QoQ_pct"] >= 0 else "#C0392B"
            ax2.text(i, fc_vals.iloc[i] * 0.02,
                     f"{row['QoQ_pct']:+.1f}%",
                     ha="center", va="bottom", fontsize=7.5,
                     color=color, fontweight="bold")

    ax2.set_xticks(list(x))
    ax2.set_xticklabels(fc_labels, fontsize=8)
    ax2.set_title(f"예측 구간  |  총 누적 증감률: {res['total_growth']:+.2f}%", fontsize=11)
    ax2.set_ylabel(f"Revenue ({unit})", fontsize=10)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    ax2.grid(axis="y", linestyle="--", alpha=0.4)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()
    print(f"[{ticker}] 누적 증감률: {res['total_growth']:+.2f}%  |  기준값: {res['base_value']/div:.2f}{unit}")

print("[OK] plot_ticker_forecast 함수 정의 완료")


### Cell 8 실행 — 차트 출력

In [ ]:
# TEST_TICKER / TEST_MODEL 은 Cell 5 에서 설정한 값을 그대로 사용
# 다른 티커/모델을 보려면 아래 변수만 변경하세요
CHART_TICKER = "AAPL"
CHART_MODEL  = "Ensemble"

plot_ticker_forecast(CHART_TICKER, CHART_MODEL)


## Cell 9 · Best-N 성장률 비교 막대 차트

향후 **4분기** / **8분기** 누적 매출 성장률을 기업별로 비교합니다.  
각 막대에 성장률(%) 값이 표시됩니다.


In [ ]:
def plot_best_n_growth(
    best_df: pd.DataFrame,
    model: str,
    forecast_date: str,
    top_n: int = None,          # None 이면 best_df 전체 사용
    figsize_per_bar: float = 0.55,
):
    """
    Best-N 기업의 4Q / 8Q 누적 성장률을 나란히 비교하는 막대 차트.

    Parameters
    ----------
    best_df       : get_best_n() 반환값
    model         : 모델명 (차트 제목용)
    forecast_date : 예측일 (차트 제목용)
    top_n         : 표시할 기업 수 (None → 전체)
    """
    df = best_df.copy()
    if top_n:
        df = df.head(top_n)

    # 4Q / 8Q 둘 다 None 이 아닌 행만 사용
    has_4q = df["growth_4q_pct"].notna().any()
    has_8q = df["growth_8q_pct"].notna().any()

    n_plots = int(has_4q) + int(has_8q)
    if n_plots == 0:
        print("[WARN] 4Q / 8Q 성장률 데이터가 없습니다.")
        return

    tickers = df["ticker"].tolist()
    n_bars  = len(tickers)
    figw    = max(14, n_bars * figsize_per_bar * n_plots)
    fig, axes = plt.subplots(1, n_plots, figsize=(figw, 7))
    if n_plots == 1:
        axes = [axes]

    fig.suptitle(
        f"매출 성장률 Best-{n_bars}  |  model: {model}  |  forecast_date: {forecast_date}",
        fontsize=14, fontweight="bold"
    )

    colors = (BAR_PALETTE * 5)[:n_bars]

    def _draw_bar(ax, values, title):
        vals  = [v if v is not None else 0 for v in values]
        bars  = ax.bar(range(n_bars), vals, color=colors, alpha=0.88,
                       width=0.65, edgecolor="white")
        # 값 레이블
        for bar, v, orig in zip(bars, vals, values):
            if orig is None:
                continue
            ypos = bar.get_height() + (max(vals) * 0.015 if v >= 0 else -max(vals) * 0.04)
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                ypos,
                f"{v:+.1f}%",
                ha="center", va="bottom", fontsize=8, fontweight="bold",
                color="#27AE60" if v >= 0 else "#C0392B"
            )
        ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
        ax.set_xticks(range(n_bars))
        ax.set_xticklabels(tickers, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("누적 성장률 (%)", fontsize=10)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plot_idx = 0
    if has_4q:
        _draw_bar(axes[plot_idx], df["growth_4q_pct"].tolist(), "향후 4분기 누적 성장률")
        plot_idx += 1
    if has_8q:
        _draw_bar(axes[plot_idx], df["growth_8q_pct"].tolist(), "향후 8분기 누적 성장률")

    plt.tight_layout()
    plt.show()

print("[OK] plot_best_n_growth 함수 정의 완료")


### Cell 9 실행 — Best-N 성장률 비교 차트 출력

In [ ]:
# Cell 7 에서 구한 best_df / used_fd 를 그대로 사용합니다.
# top_n 을 지정하면 상위 N개만 표시 (None 이면 전체)
plot_best_n_growth(
    best_df       = best_df,
    model         = BEST_MODEL,
    forecast_date = used_fd,
    top_n         = None,       # ← 원하는 수로 변경 가능 (예: 10)
)
